In [21]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
pip install dagshub mlflow --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 100.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import dagshub
import mlflow
import mlflow.sklearn

dagshub.init(repo_owner='Nestor-Dzadzamia', repo_name='ML-Advanced-Regression-Techniques', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=25b504a1-cc51-43e6-a37e-8edc56a3f396&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=a8961300f4340fec70d0037a47c7bc2d29563246852b61b96b46dac3d4905a49




Accessing as Nestor-Dzadzamia

Initialized MLflow to track repo "Nestor-Dzadzamia/ML-Advanced-Regression-Techniques"

Repository Nestor-Dzadzamia/ML-Advanced-Regression-Techniques initialized!

In [25]:
artifacts_run_id = "e55c3858ebc0409d89d07aa6e6431c56"
artifact_path = mlflow.artifacts.download_artifacts(
    run_id=artifacts_run_id,
    artifact_path="inference_artifacts.pkl"
)
with open(artifact_path, "rb") as f:
    artifacts = pickle.load(f)

print("Artifacts loaded!")

Artifacts loaded!


In [26]:
test_df = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")
ids = test_df["Id"]

test_df = test_df.drop(columns=['PoolQC', 'MiscFeature', 'Alley', 'Fence'], errors='ignore')
test_df = test_df.drop(columns=artifacts['zero_dominant'], errors='ignore')
test_df = test_df.drop(columns=artifacts['dominant_cats'], errors='ignore')
test_df[artifacts['num_cols']] = test_df[artifacts['num_cols']].fillna(artifacts['train_medians'])

for col in artifacts['cat_cols']:
    test_df[col] = test_df[col].fillna(artifacts['train_modes'][col])
test_df['TotalSF'] = test_df['TotalBsmtSF'] + test_df['1stFlrSF'] + test_df['2ndFlrSF']
test_df['TotalBath'] = (test_df['FullBath'] + test_df['BsmtFullBath'] + 
                        0.5 * test_df['HalfBath'] + 0.5 * test_df['BsmtHalfBath'])
test_df['HouseAge'] = test_df['YrSold'] - test_df['YearBuilt']
test_df['Remodeled'] = (test_df['YearRemodAdd'] != test_df['YearBuilt']).astype(int)

for col in artifacts['woe_columns']:
    test_df[col] = test_df[col].map(artifacts['target_means'][col])
test_df = pd.get_dummies(test_df, columns=artifacts['one_hot_columns'], drop_first=True)
test_df = test_df.reindex(columns=artifacts['train_columns'], fill_value=0)
test_df = test_df[artifacts['selected_cols']]


best_run_id = "a4a93017e5d1432490fa8ccfb27d6357"
model_path = mlflow.artifacts.download_artifacts(
    run_id=best_run_id,
    artifact_path="model.skops"
)
best_model = sio.load(model_path, trusted=[
    "xgboost.sklearn.XGBRegressor",
    "xgboost.core.Booster"
])
predictions = best_model.predict(test_df)
submission = pd.DataFrame({
    "Id": ids,
    "SalePrice": predictions
})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
print(submission.head())

(1459, 2)
     Id      SalePrice
0  1461  112574.914062
1  1462  135046.734375
2  1463  165470.984375
3  1464  197030.453125
4  1465  189906.437500
